# Structured Output Prompt Agent — `@azure/ai-projects`

This notebook demonstrates how to create an agent with structured output (a desired JSON schema for the response), create a conversation, generate responses using the agent, and clean up resources.

It mirrors the [`agentStructureOutput.ts`](./agentStructureOutput.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_MODEL_NAME`.

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
// Imports and configuration
import { DefaultAzureCredential } from "@azure/identity";
import { AIProjectClient } from "@azure/ai-projects";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const deploymentName = process.env["FOUNDRY_MODEL_NAME"] ?? "<model deployment name>";
console.log(`Model: ${deploymentName}`);

Model: gpt-5.2


In [2]:
// Create the AI Project client and the OpenAI client
const project = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());
// Annotated as `any` so tslab does not try to emit a non-portable declaration
// referencing the deep `node_modules/openai` (pnpm junction) path.
const openAIClient: any = project.getOpenAIClient();

In [3]:
// Create agent with structured output configuration
console.log("Creating agent...");
// Define the JSON schema for calendar events
const calendarEventSchema = {
  type: "object",
  properties: {
    name: { type: "string" },
    date: { type: "string", description: "Date in YYYY-MM-DD format" },
    participants: { type: "array", items: { type: "string" } },
  },
  required: ["name", "date", "participants"],
  additionalProperties: false,
};

const agent = await project.agents.createVersion("my-agent-structured-output", {
  kind: "prompt",
  model: deploymentName,
  text: {
    format: {
      type: "json_schema",
      name: "CalendarEvent",
      schema: calendarEventSchema,
    },
  },
  instructions: `
    You are a helpful assistant that extracts calendar event information from the input user messages,
    and returns it in the desired structured output format.
  `,
});
console.log(`Agent created (id: ${agent.id}, name: ${agent.name}, version: ${agent.version})`);

Creating agent...
Agent created (id: my-agent-structured-output:1, name: my-agent-structured-output, version: 1)


In [4]:
// Create conversation with initial user message
console.log("Creating conversation with initial user message...");
const conversation = await openAIClient.conversations.create({
  items: [
    {
      type: "message",
      role: "user",
      content: "Alice and Bob are going to a science fair this Friday, November 7, 2025.",
    },
  ],
});
console.log(`Created conversation with initial user message (id: ${conversation.id})`);

Creating conversation with initial user message...
Created conversation with initial user message (id: conv_951140ecb51c90e900nqboVgNvC0pxjd1BiLIFDPQ20FvawNDb)


In [5]:
// Generate response using the agent
console.log("Generating response...");
const response = await openAIClient.responses.create(
  {
    conversation: conversation.id,
  },
  {
    body: { agent_reference: { name: agent.name, type: "agent_reference" } },
  },
);
console.log(`Response output: ${JSON.stringify(response, null, 2)}`);

Generating response...
Response output: {
  "metadata": {},
  "top_logprobs": 0,
  "temperature": 1,
  "top_p": 0.98,
  "service_tier": "default",
  "prompt_cache_retention": "in_memory",
  "model": "gpt-5.2",
  "reasoning": {
    "effort": "none",
    "context": "current_turn",
    "mode": "standard"
  },
  "background": false,
  "text": {
    "format": {
      "type": "json_schema",
      "name": "CalendarEvent",
      "schema": {
        "type": "object",
        "properties": {
          "name": {
            "type": "string"
          },
          "date": {
            "type": "string",
            "description": "Date in YYYY-MM-DD format"
          },
          "participants": {
            "type": "array",
            "items": {
              "type": "string"
            }
          }
        },
        "required": [
          "name",
          "date",
          "participants"
        ],
        "additionalProperties": false
      },
      "strict": true
    },
    "verbosity":

In [6]:
// Clean up
console.log("Cleaning up resources...");
await openAIClient.conversations.delete(conversation.id);
console.log("Conversation deleted");

await project.agents.deleteVersion(agent.name, agent.version);
console.log("Agent deleted");

Cleaning up resources...
Conversation deleted
Agent deleted
